# Hands-On AI for Science
## August 14, Morning: A 15-Minute Introduction to PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhasegaw/hands_on_ai_for_science/blob/main/lectures/colab/a14am_torch_primer_colab.ipynb)

**No Python installed?** Click the badge above to open a self-contained version of this notebook in Google Colab.  It runs in your browser, with nothing to download and nothing to install.

The rest of today's sessions use **PyTorch**, the most widely used framework for building and running neural networks.  The good news: everything PyTorch does was already done *by hand* yesterday. We showed you how to calculate gradients via the chain rule, and a stochastic gradient descent loop.  PyTorch automates those two things, so nothing here is new math, just new programming language syntax for calling the functions that do those things.

## Setup

The cell below checks that PyTorch is installed (it is included in the pre-workshop setup instructions).  If it prints MISSING, install it now — open a terminal and run `pip install torch` — then **restart the kernel** (Kernel → Restart Kernel) and re-run from the top.

In [4]:
import importlib.util

for pkg in ['numpy', 'torch']:
    print(f'{pkg:10s}', 'OK' if importlib.util.find_spec(pkg) is not None else 'MISSING')

numpy      OK
torch      OK



1. [Tensors](#tensors)
1. [Automatic differentiation](#autograd)
1. [The training loop](#loop)
1. [Why zero_grad matters](#zerograd)
1. [Modules: containers for parameters](#modules)
1. [Names worth recognizing](#names)

<a id="tensors"></a>

## 1. Tensors

A **tensor** is PyTorch's version of a numpy array.  Creation, indexing, shapes, and broadcasting all work the way they do in numpy, and converting between the two is trivial:

In [3]:
import numpy as np
import torch

x_np = np.linspace(-np.pi, np.pi, 2000)     # a familiar numpy array
x = torch.tensor(x_np)                      # ...as a torch tensor
y = torch.sin(x)

print(x.shape, x.dtype)
print('broadcasting works the same:', (2 * x[:4] + 1).numpy())   # .numpy() converts back

torch.Size([2000]) torch.float64
broadcasting works the same: [-5.28318531 -5.27689898 -5.27061265 -5.26432632]


So why does a second numpy exist?  One reason above all: **a tensor can remember how it was computed.**  When a tensor is marked as needing gradients, PyTorch records every operation performed on it — multiplied by this, then squared, then summed — building a step-by-step history of the computation (the *computational graph*).  Every recorded step is simple (add, multiply, cosine, ...), so every step's derivative is known.  Differentiating a chain of simple steps is exactly what the chain rule does — so when gradients are requested later, PyTorch walks the recorded history backwards, applying the chain rule one step at a time.  The next section shows this in action.

<a id="autograd"></a>

## 2. Automatic differentiation

Yesterday morning, the gradient of the loss with respect to the cubic's four coefficients was derived by hand with the chain rule, and computed in numpy.  Here is that computation again, at the guessed values $a=0,\ b=1,\ c=0,\ d=-0.1$:

In [6]:
f = x_np - 0.1 * x_np**3          # yesterday's hand-built approximation of sin(x)
error = f - np.sin(x_np)
N = len(x_np)

hand = [np.sum((2/N) * error * x_np**k) for k in range(4)]
for name, g in zip('abcd', hand):
    print(f'dL/d{name} = {g:.6f}   (chain rule, by hand)')

dL/da = -0.000000   (chain rule, by hand)
dL/db = 0.683158   (chain rule, by hand)
dL/dc = -0.000000   (chain rule, by hand)
dL/dd = 3.755542   (chain rule, by hand)


Now the same four gradients, from PyTorch.  The method has three steps:
1) mark the parameters with `requires_grad=True` ("record what happens to this")
2) compute the loss
3) call `loss.backward()` ("apply the chain rule to everything recorded").

Each parameter's gradient then appears in its `.grad`:

In [7]:
a = torch.tensor(0.0,  requires_grad=True)
b = torch.tensor(1.0,  requires_grad=True)
c = torch.tensor(0.0,  requires_grad=True)
d = torch.tensor(-0.1, requires_grad=True)

y_pred = a + b*x + c*x**2 + d*x**3
loss = ((y_pred - y)**2).mean()
loss.backward()

for name, p in zip('abcd', [a, b, c, d]):
    print(f'dL/d{name} = {p.grad.item():.6f}   (autograd)')

dL/da = 0.000000   (autograd)
dL/db = 0.683158   (autograd)
dL/dc = 0.000000   (autograd)
dL/dd = 3.755542   (autograd)


**Identical, to six decimal places.**  `backward()` is not doing anything mysterious — it is applying the chain rule, mechanically, exactly as was done yesterday.  That mechanical application is what the word *backpropagation* refers to.

Importantly, the chain rule was derivable by hand for 4 parameters.  The model in the next notebook has hundreds of thousands of parameters, and production models have billions.  Automatic differentiation is the single feature that makes such models trainable — and it is the main service a deep learning framework provides.

<a id="loop"></a>

## 3. The training loop

Yesterday's numpy SGD loop, translated into PyTorch.  The structure is three steps, repeated:
1) **forward** (compute predictions and loss)
2) **backward** (compute gradients)
3) **step** (move the parameters).

An `optimizer` object does the moving — it is handed the list of parameters once, and `opt.step()` nudges each one using its `.grad`:

In [8]:
torch.manual_seed(0)
a, b, c, d = [torch.randn(1, requires_grad=True) for _ in range(4)]

opt = torch.optim.SGD([a, b, c, d], lr=0.002)

for t in range(2000):
    y_pred = a + b*x + c*x**2 + d*x**3      # forward
    loss = ((y_pred - y)**2).mean()
    opt.zero_grad()                          # clear old gradients (see next section!)
    loss.backward()                          # backward
    opt.step()                               # move the parameters
    if t % 500 == 0:
        print(t, round(loss.item(), 4))

print(f'final loss: {loss.item():.4f}   b={b.item():.3f}  d={d.item():.3f}')

0 107.8999
500 0.377
1000 0.0655
1500 0.0146
final loss: 0.0061   b=0.837  d=-0.091


The same descent as yesterday, arriving at the same kind of answer ($b \approx 0.84$, $d \approx -0.09$, loss heading toward the familiar 0.004 floor).  Two things changed: no gradient formulas appear anywhere (autograd derived them), and the update rule lives in the optimizer.  That second change is very convenient. Swapping `SGD` for a smarter update rule like `Adam` is a one-word edit, and the transformer trained in the next notebook does exactly that.


<a id="zerograd"></a>

## 4. Why zero_grad matters

One common mistake in PyTorch use is responsible for more silent PyTorch bugs than anything else.  **Gradients accumulate.**  Each call to `backward()` *adds* to `.grad` rather than replacing it — by design, to support advanced uses.  Forgetting to clear them between steps mixes yesterday's gradient into today's:

In [9]:
b = torch.tensor(1.0, requires_grad=True)

loss = ((0 + b*x - 0.1*x**3 - y)**2).mean()
loss.backward()
print('after one backward: ', round(b.grad.item(), 6))

loss = ((0 + b*x - 0.1*x**3 - y)**2).mean()
loss.backward()
print('after two backwards:', round(b.grad.item(), 6), '  <- doubled, NOT recomputed')

b.grad = None    # what opt.zero_grad() does for every parameter
loss = ((0 + b*x - 0.1*x**3 - y)**2).mean()
loss.backward()
print('after clearing:     ', round(b.grad.item(), 6))

after one backward:  0.683158
after two backwards: 1.366316   <- doubled, NOT recomputed
after clearing:      0.683158


A model trained with this bug still runs, still prints falling losses, and quietly converges worse — the hardest kind of error to notice.  Hence the ritual: **`opt.zero_grad()` at the top of every loop iteration**.

<a id="modules"></a>

## 5. Modules: containers for parameters

Real models have too many parameters to name one by one, so PyTorch packages them.  An `nn.Module` is a container of parameters; `nn.Linear(3, 1)` holds a weight for each of 3 inputs plus a bias — which is exactly the cubic model, viewed as a *linear* function of the features $(x, x^2, x^3)$:

In [11]:
import torch.nn as nn

hidden_layer_size = 5


features = torch.stack([x, x**2, x**3], dim=1).float()   # (2000, 3): columns x, x^2, x^3


targets = y.float()

model = nn.Linear(3, 1)                       # weights for x, x^2, x^3, plus a bias (= a)
opt = torch.optim.SGD(model.parameters(), lr=0.002)

for t in range(2000):
    y_pred = model(features).squeeze()
    loss = ((y_pred - targets)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

w = model.weight.detach().numpy().ravel()
print(f'loss {loss.item():.4f} | a={model.bias.item():.3f} b={w[0]:.3f} c={w[1]:.3f} d={w[2]:.3f}')

loss 0.0047 | a=0.003 b=0.835 c=-0.000 d=-0.090


Note what `model.parameters()` did: it handed *all* of the module's parameters to the optimizer in one call, with no need to list them.  Every PyTorch model — including the transformer in the next notebook — is a nesting of modules inside modules, trained by this exact loop: forward, loss, `zero_grad`, `backward`, `step`.

<a id="names"></a>

## 6. Names worth recognizing

Three more pieces of vocabulary appear in the next notebook:

* **`Dataset` and `DataLoader`** — packaging for data.  A `Dataset` knows how to fetch one example; a `DataLoader` serves examples in shuffled minibatches.  Together they are yesterday's "choose a random minibatch" step.
* **`.to(device)`** — one line that moves a model or tensor onto other hardware (a GPU, or Apple's `mps`).  Everything this week runs comfortably on plain CPU, so the line will appear but never matter here.

And that is the whole primer.  The transformer notebook that follows contains no PyTorch machinery beyond what is on this page — tensors, autograd, optimizers, modules, and data loaders — just composed at larger scale.